# 04 - XLM-R fine-tuning on Colab (Part 6A: page-level threat classification)

The **NLP-only arm** of C2's ablation study: Part 5A `page_text` → fine-tuned `xlm-roberta-base` → 5 threat classes, on the **shared** v0.3 split (same 68 test pages as DOM-only and Behavior-only).

This notebook needs **only the bundle zip** made on the laptop - not the repo, not the raw pages:

```
python scripts/train_xlmr.py export-bundle --version v0.3   # -> data/tmp/xlmr_bundle_v0.3.zip
```

**Steps:** 1 GPU runtime → 2 upload bundle → 3 install pinned packages → 4 (optional) smoke check → 5 train 3 seeds → 6 look at VAL results → 7 zip + download → back on the laptop: `import-model` + `evaluate_xlmr.py`.

Expected on a free **T4**: ~3-5 min per seed (early stopping, usually < 10 epochs), ~15-20 min in total including installs and the 1.1 GB model download.

*Test is used once:* training and the seed selection only use VAL. Test probabilities are written after the selection and scored on the laptop.

## 1. GPU runtime
**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.** Then run the cell below - it must print a GPU name.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

## 2. Upload the bundle
Choose `xlmr_bundle_v0.3.zip` (from `data/tmp/` on the laptop, ~0.4 MB). It is unpacked to `/content/bundle`.

In [ ]:
import os, zipfile
from google.colab import files

VERSION = "v0.3"
uploaded = files.upload()  # pick data/tmp/xlmr_bundle_v0.3.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall("/content/bundle")
os.chdir("/content/bundle")
print(open("README.txt").read())

## 3. Install pinned packages
`requirements-colab.txt` pins transformers / tokenizers / accelerate / safetensors / sentencepiece to the **laptop's** versions, so the laptop can load and re-check the model exactly. Colab's own CUDA build of torch is kept. (A pip warning about other Colab packages is harmless.)

In [ ]:
!pip -q install -r requirements-colab.txt
import transformers, accelerate, torch
print("transformers", transformers.__version__, "| accelerate", accelerate.__version__, "| torch", torch.__version__)

## 4. (Optional) smoke check - 1 training step with the real model
Downloads `xlm-roberta-base` (1.1 GB, once) and runs one step on a tiny subset. Output goes to `smoke_out/` (thrown away).

In [ ]:
!python run_train.py --smoke --seeds 42 --out smoke_out

## 5. Train 3 seeds
Settings come from `configs/xlmr.yaml` (lr 2e-5, ≤ 10 epochs, batch 16, warmup 0.1, weight decay 0.01, fp16, class-weighted loss, early stopping on VAL macro-F1 with patience 3). Each seed prints its best VAL macro-F1, epochs, **time** and the **GPU name**.

**Out of memory?** In `configs/xlmr.yaml` set `batch_size: 8` and `gradient_accumulation: 2` (same effective batch) and re-run this cell.

In [ ]:
import time
started = time.time()
!python run_train.py --out models/xlmr/{VERSION}
print(f"total: {(time.time() - started) / 60:.1f} min")

## 6. VAL results (test is NOT looked at here)

In [ ]:
import json
card = json.load(open(f"models/xlmr/{VERSION}/model_card.json"))
for seed, row in card["seeds"].items():
    print(f"seed {seed}: val macro-F1 {row['val_macro_f1']:.3f}  best epoch {row['best_epoch']}  "
          f"{row['epochs_run']} epochs  {row['training_seconds']:.0f} s")
print("final model = seed", card["selection"]["final_seed"], "|", card["gpu_name"])

## 7. Zip + download
Default (**light**, ~1.1 GB): `final/` with weights + every seed's metrics and val/test predictions (enough for the 3-seed mean ± std). `FULL = True` also keeps every seed's weights (~4.4 GB) - then copying to Google Drive is easier than a browser download.

In [ ]:
import shutil, zipfile
from pathlib import Path

FULL = False
root = Path(f"models/xlmr/{VERSION}")
out = Path(f"/content/xlmr_{VERSION}.zip")
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as z:  # weights don't compress
    for path in sorted(root.rglob("*")):
        if path.is_dir():
            continue
        rel = path.relative_to(root.parent)
        is_seed_weight = path.parent.name.startswith("seed_") and path.name == "model.safetensors"
        if is_seed_weight and not FULL:
            continue
        z.write(path, rel.as_posix())
print(out, f"{out.stat().st_size / 1e9:.2f} GB")
files.download(str(out))
# Alternative for big files:
# from google.colab import drive; drive.mount("/content/drive"); shutil.copy(out, "/content/drive/MyDrive/")

## 8. Back on the laptop
```
python scripts/train_xlmr.py import-model --zip <Downloads>/xlmr_v0.3.zip --version v0.3
python scripts/evaluate_xlmr.py --version v0.3
```
`import-model` refuses a zip whose split fingerprint or label mapping differs from the laptop's. `evaluate_xlmr.py` re-runs the final model on the CPU, checks it agrees with the Colab predictions, scores val + test once (per language, macro P/R/F1, confusion matrices, 3-seed mean ± std) and prints the ablation table.